# Моделирование языка

**AI3 · NLP Deep Dive** — [AI Olympiad Curriculum](https://ai-olympiad-curriculum.netlify.app)

~15 min

## Введение

Семантические векторы, такие как Word2Vec и GloVe, на самом деле являются первым шагом к **моделированию языка** — созданию моделей, которые каким-то образом *понимают* (или *представляют*) природу языка.


Основная идея моделирования языка заключается в обучении моделей на неразмеченных данных в режиме обучения без учителя. Это важно, потому что у нас есть огромные объемы неразмеченного текста, в то время как количество размеченного текста всегда будет ограничено усилиями, которые мы можем потратить на разметку. Чаще всего мы можем создавать языковые модели, которые способны **предсказывать пропущенные слова** в тексте, так как легко замаскировать случайное слово в тексте и использовать его как обучающий пример.



## Обучение векторов

В предыдущих примерах мы использовали заранее обученные семантические векторы, но интересно понять, как эти векторы можно обучать. Существует несколько подходов:

* **Моделирование языка с помощью N-грамм**, когда мы предсказываем токен, основываясь на N предыдущих токенах (N-граммы).
* **Непрерывный мешок слов** (Continuous Bag-of-Words, CBoW), когда мы предсказываем центральный токен $W_0$ в последовательности токенов $W_{-N}$, ..., $W_N$.
* **Skip-gram**, где мы предсказываем набор соседних токенов {$W_{-N},\dots, W_{-1}, W_1,\dots, W_N$} на основе центрального токена $W_0$.

![изображение из статьи о преобразовании слов в векторы](ms-assets/ai/example-algorithms-for-converting-words-to-vectors.fbe9207a726922f6.webp)

> Изображение из [этой статьи](https://arxiv.org/pdf/1301.3781.pdf)



## Пример ноутбуков: Обучение модели CBoW

Продолжите обучение с помощью следующих ноутбуков:

* [Обучение CBoW Word2Vec с использованием TensorFlow](https://github.com/microsoft/AI-For-Beginners/blob/main/lessons/5-NLP/15-LanguageModeling/CBoW-TF.ipynb)
* [Обучение CBoW Word2Vec с использованием PyTorch](https://github.com/microsoft/AI-For-Beginners/blob/main/lessons/5-NLP/15-LanguageModeling/CBoW-PyTorch.ipynb)



## Заключение

В предыдущем уроке мы увидели, что векторы слов работают как магия! Теперь мы знаем, что обучение векторов слов — это не такая уж сложная задача, и мы можем обучить свои собственные векторы для текстов, специфичных для определенной области, если это потребуется.



## Обзор и самостоятельное изучение

* [Официальный учебник PyTorch по моделированию языка](https://pytorch.org/tutorials/beginner/nlp/word_embeddings_tutorial.html).
* [Официальный учебник TensorFlow по обучению модели Word2Vec](https://www.TensorFlow.org/tutorials/text/word2vec).
* Использование фреймворка **gensim** для обучения наиболее популярных векторов всего за несколько строк кода описано [в этой документации](https://pytorch.org/tutorials/beginner/nlp/word_embeddings_tutorial.html).



## Задание: Обучите модель Skip-Gram

В лабораторной работе мы предлагаем вам изменить код из этого урока, чтобы обучить модель Skip-Gram вместо CBoW. [Прочитайте подробности](https://github.com/microsoft/AI-For-Beginners/blob/main/lessons/5-NLP/15-LanguageModeling/lab/README.md)



---

## ✅ Self-check tasks

Implement the function in the 'your code' cell, then run the check cell — it runs randomized tests and shows the first mismatch. Run the helper cell below once first.

In [ ]:
# ── task-checking helper cell — run once ──
import copy
import numpy as np
import math, statistics, itertools, collections, functools, heapq, random

try:
    import sys; sys.stdout.reconfigure(encoding="utf-8", errors="replace")
except Exception:
    pass

_L = {"noFn":"❓ Define a function `{e}` in the solution cell and run it","runErr":"❌ Test {i}: {t}: {m}","wrong":"❌ Test {i}: {r}","badRef":"⚠️ Check harness broke (generator/reference): {t}: {m}","inL":"Input","expL":"Expected","gotL":"Got","ok":"✅ Correct! All {n} tests passed."}

def _nb_copy_kw(kw):
    return {k: (v.copy() if isinstance(v, np.ndarray) else copy.deepcopy(v)) for k, v in kw.items()}

def _nb_jsafe(x, depth=0):
    if depth > 6: return "…"
    if isinstance(x, np.ndarray): return _nb_jsafe(x.tolist(), depth + 1)
    if isinstance(x, np.integer): return int(x)
    if isinstance(x, np.floating): return float(x)
    if isinstance(x, np.bool_): return bool(x)
    if isinstance(x, float): return round(x, 6) if math.isfinite(x) else str(x)
    if isinstance(x, (list, tuple)): return [_nb_jsafe(v, depth + 1) for v in x]
    if isinstance(x, dict): return {str(k): _nb_jsafe(v, depth + 1) for k, v in x.items()}
    if isinstance(x, (int, str, bool)) or x is None: return x
    return repr(x)

def _nb_cmp(exp, got, path=""):
    if isinstance(exp, np.ndarray) or isinstance(got, np.ndarray):
        e, g = np.asarray(exp), np.asarray(got)
        if e.shape != g.shape:
            return f"shape {tuple(g.shape)} != expected {tuple(e.shape)}"
        try:
            if np.issubdtype(e.dtype, np.number) or np.issubdtype(g.dtype, np.number):
                ef, gf = e.astype(float), g.astype(float)
                close = np.isclose(gf, ef, atol=1e-6, rtol=1e-5, equal_nan=True)
                if not bool(close.all()):
                    i = tuple(int(v) for v in np.argwhere(~close)[0])
                    return f"{path}first mismatch at index {i}: got {gf[i]:.6g} vs expected {ef[i]:.6g}"
            elif not np.array_equal(g, e):
                return f"{path}values differ"
        except Exception as ex:
            return f"{path}cannot compare: {ex}"
        return None
    if isinstance(exp, (int, float)) and not isinstance(exp, bool) and isinstance(got, (int, float)) and not isinstance(got, bool):
        if not np.isclose(float(exp), float(got), atol=1e-6, rtol=1e-5):
            return f"{path}{got} != expected {exp}"
        return None
    if isinstance(exp, (list, tuple)) and isinstance(got, (list, tuple)):
        if len(exp) != len(got):
            return f"{path}length {len(got)} != expected {len(exp)}"
        for i, (a, b) in enumerate(zip(exp, got)):
            r = _nb_cmp(a, b, path + f"[{i}].")
            if r: return r
        return None
    if isinstance(exp, dict) and isinstance(got, dict):
        if set(exp) != set(got):
            return f"{path}keys differ"
        for k in exp:
            r = _nb_cmp(exp[k], got[k], path + f"[{k!r}].")
            if r: return r
        return None
    if exp != got:
        return f"{path}{got!r} != expected {exp!r}"
    return None

def _check_task(g, ref_src, entry, n=20, seed=0):
    """g — globals() ноутбука; ref_src — gen() + эталонное решение."""
    fn = g.get(entry)
    if not callable(fn):
        print(_L["noFn"].format(e=entry)); return
    ns = {"np": np, "math": math, "statistics": statistics, "itertools": itertools,
          "collections": collections, "functools": functools, "heapq": heapq, "random": random}
    try:
        exec(ref_src, ns)
        gen, ref = ns["gen"], ns[entry]
    except Exception as e:
        print(_L["badRef"].format(t=type(e).__name__, m=e)); return
    rng = np.random.default_rng(seed)
    for i in range(n):
        try:
            kw = gen(rng)
        except Exception as e:
            print(_L["badRef"].format(t=type(e).__name__, m=e)); return
        try:
            got = fn(**_nb_copy_kw(kw))
        except Exception as e:
            print(_L["runErr"].format(i=i + 1, t=type(e).__name__, m=e)
                  + "\n" + _L["inL"] + ": " + repr(_nb_jsafe(kw))); return
        exp = ref(**_nb_copy_kw(kw))
        r = _nb_cmp(exp, got)
        if r:
            print(_L["wrong"].format(i=i + 1, r=r)
                  + "\n" + _L["inL"] + ": " + repr(_nb_jsafe(kw))
                  + "\n" + _L["expL"] + ": " + repr(_nb_jsafe(exp))
                  + "\n" + _L["gotL"] + ": " + repr(_nb_jsafe(got))); return
    print(_L["ok"].format(n=n))

### Task 1 — Word n-grams

●●○ · ~8 min

> Word n-grams capture local context that bag-of-words loses ('not good' ≠ 'good'). They are just sliding windows over the token list — and the building block of every statistical language model.

Implement `ngrams(tokens, n)` → list of tuples of `n` consecutive tokens. `ngrams(['a','b','c'], 2)` → `[('a','b'), ('b','c')]`. If `n` exceeds the number of tokens, return an empty list.

<details><summary>Hint</summary>

[tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)] — the range is empty when the bound is non-positive.

</details>


In [ ]:
import numpy as np

def ngrams(tokens, n):
    # your code
    return []

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    toks = [str(w) for w in rng.choice([\"a\", \"b\", \"c\", \"d\", \"e\", \"f\"],\n                                      int(rng.integers(3, 12)))]\n    return {\"tokens\": toks, \"n\": int(rng.integers(1, 5))}\n\ndef ngrams(tokens, n):\n    return [tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]", "ngrams", 20)

### Task 2 — Perplexity from log-probs

●○○ · ~6 min

> Perplexity is the standard language-model metric: the exponential of the average **negative** log-likelihood, PPL = exp(−mean(log p(token))). A perfect model assigns log p = 0 → PPL = 1; worse models get higher values.

Implement `perplexity(logprobs)`: given the natural log-probabilities the model assigned to the true tokens, return exp(−mean(logprobs)).

<details><summary>Hint</summary>

One line: math.exp(-np.mean(logprobs)). Mind the minus sign.

</details>


In [ ]:
import numpy as np

def perplexity(logprobs):
    # your code
    return float(np.exp(np.mean(logprobs)))  # wrong — sign

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    n = int(rng.integers(3, 30))\n    return {\"logprobs\": -rng.uniform(0.05, 4.0, n)}\n\ndef perplexity(logprobs):\n    return float(math.exp(-float(np.mean(logprobs))))", "perplexity", 20)

---

**Go deeper**
- [Исходный урок (Microsoft, MIT)](https://github.com/microsoft/AI-For-Beginners/blob/main/lessons/5-NLP/15-LanguageModeling)